# 🔄 Rekabetten Kooperatif Sisteme Veri Akışı: Nash to SHAP Pipeline
### *Non-Cooperative Game -> Evolutionary Filtering (ESS) -> Cooperative Game Attribution*

Bu notebook, 3 aşamalı uçtan uca akışı gösterir:
1. **Adım 1:** Veri kümesindeki öznitelikler arası rekabet matrisinin kurulması ve Nash Denge analizi.
2. **Adım 2:** Maynard Smith ESS filtresi ile veri kaymalarına ve gürültüye dayanıksız (kırılgan) özniteliklerin elenmesi.
3. **Adım 3:** Kalan sağlam öznitelikler üzerinde eğitilen modelin SHAP motoruyla açıklanması.

In [ ]:
import numpy as np
import xgboost as xgb
import shap
from core_theory.nash_equilibrium import FeaturePayoffMatrix, compute_nash_equilibrium, detect_feature_suppression, NashFeatureRanker
from core_theory.maynard_ess_filter import MaynardESSFilter
from data_ops.synthetic_coalitions import generate_coalition_dataset
from explainers.waterfall_local import render_waterfall
from explainers.beeswarm_global import render_summary

# Sentetik veri oluşturma (Simetrik, Sinerjik, Doğrusal ve Dummy özellikler)
dataset = generate_coalition_dataset(n_samples=1000, random_state=42)
X, y = dataset.X, dataset.y
feature_names = dataset.feature_names
print(f'Veri kümesi oluşturuldu: {X.shape}, Öznitelikler: {feature_names}')

### Aşama 1: Nash Rekabet ve Baskılama Analizi

In [ ]:
payoff_matrix = FeaturePayoffMatrix.from_correlations(X, y, feature_names=feature_names)
ranker = NashFeatureRanker(payoff_matrix)
ranked_features = ranker.rank()
print('Nash Dengesi Sıralaması:')
for r in ranked_features:
    print(f"- {r['feature']:<15} | Nash Ağırlığı: {r['nash_weight']:.4f} | İzole Getiri: {r['isolated_payoff']:.4f}")

### Aşama 2: Maynard Smith ESS Filtresi (Gürültü ve Mutasyon Direnci)

In [ ]:
ess_filter = MaynardESSFilter(n_generations=80, mutation_rate=0.2, random_state=42)
ess_result = ess_filter.fit_filter(X, y, feature_names=feature_names)
print(f"Seçilen Sağlam (ESS) Öznitelikler: {ess_result['selected_features']}")
print(f"Elenen Kırılgan / Kukla Öznitelikler: {ess_result['eliminated_features']}")

### Aşama 3: Kooperatif Oyun (SHAP Açıklaması)

In [ ]:
selected_indices = ess_result['selected_indices']
X_robust = X[:, selected_indices]
names_robust = ess_result['selected_features']

model = xgb.XGBRegressor(n_estimators=80, max_depth=4, learning_rate=0.1, random_state=42)
model.fit(X_robust, y)

explainer = shap.TreeExplainer(model)
shap_values = explainer(X_robust)

render_waterfall(shap_values[0])
render_summary(shap_values)